In [16]:
import pandas as pd
import numpy as np
import re

from pathlib import Path

In [17]:
ROOT = Path.cwd().parent

DATASETS = ROOT / "datasets"
MODELS = ROOT / "models"

INTERACTIONS = DATASETS / "eedi_interactions"
MISCONCEPTIONS = DATASETS / "eedi_misconceptions"

PROCESSED = DATASETS / "processed"

PROCESSED.mkdir(exist_ok=True)

print("Processed data folder:", PROCESSED)

Processed data folder: c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet\datasets\processed


In [18]:
miscon = pd.read_csv(
    MISCONCEPTIONS / "train.csv"
)

print("Original shape:", miscon.shape)

Original shape: (1869, 15)


In [19]:
rows = []

for _, row in miscon.iterrows():

    options = {
        "A": row["AnswerAText"],
        "B": row["AnswerBText"],
        "C": row["AnswerCText"],
        "D": row["AnswerDText"]
    }

    labels = {
        "A": row["MisconceptionAId"],
        "B": row["MisconceptionBId"],
        "C": row["MisconceptionCId"],
        "D": row["MisconceptionDId"]
    }

    for option in options:
        if pd.notna(labels[option]):
            rows.append({
                "QuestionId": row["QuestionId"],
                "QuestionText": row["QuestionText"],
                "SelectedAnswer": options[option],
                "CorrectAnswer": row["CorrectAnswer"],
                "MisconceptionId": int(labels[option])
            })

miscon_train = pd.DataFrame(rows)

print("Examples:", len(miscon_train))

Examples: 4370


In [20]:
def clean_math_text(text):
    text = str(text)

    # Normalize whitespace only
    text = re.sub(r"\s+", " ", text)

    return text.strip()


miscon_train["QuestionText"] = (
    miscon_train["QuestionText"]
    .apply(clean_math_text)
)

miscon_train["SelectedAnswer"] = (
    miscon_train["SelectedAnswer"]
    .apply(clean_math_text)
)

miscon_train["text"] = (
    "question: " +
    miscon_train["QuestionText"] +
    " student answer: " +
    miscon_train["SelectedAnswer"]
)

In [21]:
for i in range(3):
    print(f"\nExample {i + 1}")
    print("Question:", miscon_train.iloc[i]["QuestionText"])
    print("Answer:", miscon_train.iloc[i]["SelectedAnswer"])
    print("Label:", miscon_train.iloc[i]["MisconceptionId"])


Example 1
Question: \[ 3 \times 2+4-5 \] Where do the brackets need to go to make the answer equal \( 13 \) ?
Answer: Does not need brackets
Label: 1672

Example 2
Question: Simplify the following, if possible: \( \frac{m^{2}+2 m-3}{m-3} \)
Answer: \( m+1 \)
Label: 2142

Example 3
Question: Simplify the following, if possible: \( \frac{m^{2}+2 m-3}{m-3} \)
Answer: \( m+2 \)
Label: 143


In [22]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        miscon_train,
        groups=miscon_train["QuestionId"]
    )
)

miscon_train_data = miscon_train.iloc[train_idx].reset_index(drop=True)
miscon_val_data = miscon_train.iloc[val_idx].reset_index(drop=True)

print("Training examples:", len(miscon_train_data))
print("Validation examples:", len(miscon_val_data))

Training examples: 3503
Validation examples: 867


In [23]:
train_questions = set(miscon_train_data["QuestionId"])
val_questions = set(miscon_val_data["QuestionId"])

print("Training questions:", len(train_questions))
print("Validation questions:", len(val_questions))
print("Question overlap:", len(train_questions & val_questions))

Training questions: 1495
Validation questions: 374
Question overlap: 0


In [24]:
PROCESSED.mkdir(exist_ok=True)

print("Processed directory:", PROCESSED)

Processed directory: c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet\datasets\processed


In [25]:
train_path = PROCESSED / "misconception_train.csv"
val_path = PROCESSED / "misconception_val.csv"

miscon_train_data.to_csv(train_path, index=False)
miscon_val_data.to_csv(val_path, index=False)

print("Saved:")
print(train_path)
print(val_path)

Saved:
c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet\datasets\processed\misconception_train.csv
c:\Users\Akshat\Desktop\Over_Here\Projects\Quintet\datasets\processed\misconception_val.csv


In [26]:
train_check = pd.read_csv(train_path)
val_check = pd.read_csv(val_path)

print("TRAIN:", train_check.shape)
print("VALIDATION:", val_check.shape)

print("\nColumns:")
print(train_check.columns.tolist())

print("\nUnique misconception labels:",
      train_check["MisconceptionId"].nunique())

print("\nMissing values:")
print(train_check.isnull().sum())

TRAIN: (3503, 6)
VALIDATION: (867, 6)

Columns:
['QuestionId', 'QuestionText', 'SelectedAnswer', 'CorrectAnswer', 'MisconceptionId', 'text']

Unique misconception labels: 1414

Missing values:
QuestionId         0
QuestionText       0
SelectedAnswer     0
CorrectAnswer      0
MisconceptionId    0
text               0
dtype: int64
